# 08 · Problem detector and fusion (RQ5, RQ6 on synthetic data)

**Diagnose.** A ResNet-18 trained on Blender images predicts, from the same 256×256 crop the pose model sees:
occlusion / quality / framing probabilities, per-joint visibility, and the hidden fraction of the body.
Labels are free: they come from notebook 04's generator and from `pose_lib.degrade`.

**Fuse.** Router weights come from the detector. Fusion methods compared (RQ5):
`weighted` (one weight per expert), `per_joint` (the occlusion expert gets more say on joints predicted
hidden), `learned` (small MLP trained on a held-out split), and `top1` (hard routing), plus oracle routing.

Needs notebook 07's adapters. ~30 min on a T4.

In [ ]:
EXPERTS = ["E0", "E1", "E2", "E3"]
R_LORA, LORA_BLOCKS, SEED = 16, 4, 0
RUN = f"experts_r{R_LORA}_b{LORA_BLOCKS}_s{SEED}"           # notebook 07's run
DET_EPOCHS, DET_PER_SOURCE = 3, 3000

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# HMR2.0 without PHALP/detectron2 (the install that worked in notebook 05 on Python 3.13)
%cd /content
![ -d 4D-Humans ] || git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /content/4D-Humans
!pip install -q numpy yacs smplx pytorch_lightning timm einops omegaconf trimesh webdataset pyrender
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy
!pip install -q --no-deps .
!sed -i 's/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg)/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg, weights_only=False)/' hmr2/models/__init__.py
%cd /content
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, "/content/4D-Humans")

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models
download_models(CACHE_DIR_4DHUMANS)                           # ~2.5 GB, cached for the session

# SMPL: plain-numpy files only, chumpy never imported (the fix for the broken environment)
from pose_lib.smpl import pkl_to_npz, setup_hmr2_smpl
SMPL_NPZ = f"{PROJECT_DIR}/data/smpl_neutral.npz"
if not os.path.exists(SMPL_NPZ):
    print(pkl_to_npz(f"{PROJECT_DIR}/body_models/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", SMPL_NPZ))
setup_hmr2_smpl(SMPL_NPZ)

import torch
from pose_lib import hmr2_ft as ft, lora, metrics, stats
fk = ft.SMPLJoints(SMPL_NPZ)
print("HMR2.0 + SMPL ready")

In [ ]:
# synthetic AMASS dataset from notebook 04 (B_clean / C_paste2d / D_occ3d, train + test)
DATA = "/content/dataset"
if not os.path.exists(f"{DATA}/test/D_occ3d/labels.npz"):
    !tar -xf "{PROJECT_DIR}/data/synth_amass.tar" -C /content
# restore self-occlusion labels if notebook 06 added them in an earlier session
for c in ["B_clean", "C_paste2d", "D_occ3d"]:
    saved = f"{PROJECT_DIR}/data/test_labels_selfocc_{c}.npz"
    if os.path.exists(saved):
        shutil.copy(saved, f"{DATA}/test/{c}/labels.npz")
HAS_SELFOCC = "selfvis_joint" in np.load(f"{DATA}/test/D_occ3d/labels.npz").files
print(os.listdir(f"{DATA}/train"), "| self-occlusion labels:", HAS_SELFOCC)

In [ ]:
def fresh_model():
    m = ft.load_model()
    gc.collect(); torch.cuda.empty_cache()
    return m

In [ ]:
!pip install -q torchvision
from pose_lib import experts as ex, detector as dt, degrade, fusion
MODELS = f"{PROJECT_DIR}/models/{RUN}"; RESULTS = f"{PROJECT_DIR}/results/{RUN}"
model = fresh_model()
lora.inject(model, lora.hmr2_targets(n_blocks=LORA_BLOCKS), EXPERTS, r=R_LORA)
for e in EXPERTS:
    lora.load_adapter(model, e, f"{MODELS}/{e}.pt")
model.cuda().eval(); lora.set_active(model, None)

## Train the detector

In [ ]:
def split(cond):
    return ft.split_train_val(f"{DATA}/train/{cond}", seed=SEED)
trB, vaB = split("B_clean"); trD, vaD = split("D_occ3d"); trC, vaC = split("C_paste2d")
Q, Fr = degrade.quality(), degrade.framing()
def sources(tr_or_va, deterministic):
    iB, iD, iC = (trB, trD, trC) if tr_or_va == "train" else (vaB, vaD, vaC)
    kw = dict(deterministic=deterministic, augment=not deterministic)
    return [ft.SynthDataset(f"{DATA}/train/B_clean", indices=iB, **kw),
            ft.SynthDataset(f"{DATA}/train/D_occ3d", indices=iD, **kw),
            ft.SynthDataset(f"{DATA}/train/C_paste2d", indices=iC, **kw),
            ft.SynthDataset(f"{DATA}/train/B_clean", indices=iB, transform=Q, transform_seed=1, **kw),
            ft.SynthDataset(f"{DATA}/train/B_clean", indices=iB, transform=Fr, transform_seed=2, **kw),
            ft.SynthDataset(f"{DATA}/train/D_occ3d", indices=iD, transform=Q, transform_seed=3, **kw)]
det_train = dt.MixtureDataset(sources("train", False), [DET_PER_SOURCE] * 6, seed=SEED)
det_val = dt.MixtureDataset(sources("val", True), [400] * 6, seed=SEED)
DET_PATH = f"{MODELS}/detector_resnet18.pt"
det = dt.Detector("resnet18").cuda()
if os.path.exists(DET_PATH):
    det.load_state_dict(torch.load(DET_PATH, map_location="cuda"))
else:
    hist = dt.train(det, det_train, det_val, DET_PATH, epochs=DET_EPOCHS, device="cuda")
    json.dump(hist, open(f"{MODELS}/detector_history.json", "w"), indent=1)
    det.load_state_dict(torch.load(DET_PATH, map_location="cuda"))
det.eval()

In [ ]:
# RQ6 (synthetic half): how well does the detector diagnose held-out TEST conditions?
tests = ex.test_sets(DATA)
det_test = dt.MixtureDataset(list(tests.values()), [len(v) for v in tests.values()], seed=0)
m = dt.evaluate(det, det_test, device="cuda")
json.dump(m, open(f"{RESULTS}/detector_test_metrics.json", "w"), indent=1); m

## Learned fusion: fit on held-out validation crops (never the test set)

In [ ]:
FUS_PATH = f"{MODELS}/learned_fusion.pt"
if os.path.exists(FUS_PATH):
    learned = fusion.LearnedFusion(k=len(EXPERTS)); learned.load_state_dict(torch.load(FUS_PATH))
else:
    cache = ex.cache_for_fusion(model, det, det_val, experts=EXPERTS, device="cuda")
    learned, best = fusion.train_learned_fusion(cache, fk.cpu(), epochs=150)
    torch.save(learned.state_dict(), FUS_PATH)
learned.eval()

## RQ5: compare fusion methods on every test condition

In [ ]:
def run_eval(name, ds, tname, **kw):
    f = f"{RESULTS}/{name}__{tname}.npz"
    if os.path.exists(f):
        return dict(np.load(f))
    print(f"{name:>22} on {tname}:", end=" ")
    r = ft.evaluate(model, ds, fk, **kw); np.savez(f, **r); return r

variants = {"DGS_det_weighted": ex.DGS(model, det, "weighted"), "DGS_det_perjoint": ex.DGS(model, det, "per_joint"),
            "DGS_det_learned": ex.DGS(model, det, "learned", learned=learned), "DGS_det_top1": ex.DGS(model, det, "top1")}
R = {}
for tname, ds in tests.items():
    for name, dgs in variants.items():
        R[(name, tname)] = run_eval(name, ds, tname, predict_fn=dgs)
    for ref in ["pretrained", "GEN", "DGS_oracle_weighted", "DGS_oracle_perjoint"] + EXPERTS:
        f = f"{RESULTS}/{ref}__{tname}.npz"
        if os.path.exists(f):
            R[(ref, tname)] = dict(np.load(f))

In [ ]:
import pandas as pd
rows_ = ["pretrained", "GEN", "E0", "E1", "E2", "E3", "DGS_oracle_weighted", "DGS_oracle_perjoint"] + list(variants)
tab = pd.DataFrame({t: {m: R[(m, t)]["pa_mpjpe"].mean() for m in rows_ if (m, t) in R} for t in tests}).round(1)
tab["mean"] = tab.mean(1).round(1)
tab.to_csv(f"{RESULTS}/rq5_fusion_pa_mpjpe.csv"); tab

In [ ]:
# Is the best detector-routed variant better than the generalist? (RQ4 with a real detector)
best = tab.loc[list(variants), "mean"].idxmin(); print("best fusion:", best)
rows = []
for t in tests:
    mdiff, lo, hi = stats.paired_bootstrap(R[(best, t)]["pa_mpjpe"], R[("GEN", t)]["pa_mpjpe"])
    rows.append({"test": t, f"{best} - GEN (mm)": round(mdiff, 2), "95% CI": f"[{lo:+.1f}, {hi:+.1f}]",
                 "p": stats.bootstrap_p(R[(best, t)]["pa_mpjpe"], R[("GEN", t)]["pa_mpjpe"])})
c = pd.DataFrame(rows); c["p_holm"] = stats.holm(c.p.values).round(4)
c.to_csv(f"{RESULTS}/rq4_detector_vs_generalist.csv", index=False); c